In [1]:
# Main to test model parameter influence on the model performance

In [2]:
RandSeedNumber = 18264
nr = '005'
name = '25og'

In [3]:
import os
import numpy as np
import torch
torch.cuda.empty_cache()
import torch_geometric
from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader 
from GNNmodel_Model_BatchNorm_EdgeAttr import myPNA
from GNNmodel_Visuals import visualize_snapshot_cell, visualize_snapshot_nucleus, visualize_snapshot_graph, lin_regression
import scipy.stats
import seaborn as sns
from torch.optim.lr_scheduler import CosineAnnealingLR
from scipy import io
import pickle
import random
import copy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import networkx as nx
plt.rcParams['font.family'] = 'serif'
print(torch.cuda.is_available())

plt.rcParams['xtick.direction'] = plt.rcParams['ytick.direction'] = 'in'

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /vast.mnt/home/20234017/.local/lib/python3.11/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs
  import torch_geometric.typing


True


In [4]:
# function for loading the processed data

def load_dataset(file_name):
    with open(file_name, 'rb') as f:
        data = pickle.load(f)
        return data

In [5]:
# load data
data_list_all = load_dataset('AllProcessedData_BigGrid_batch3_[FINAL].pkl')
data_list_validation = load_dataset('AllProcessedData_BigGrid_batch3_[FINAL_VALIDATION].pkl')

In [6]:
# Reduce amount of low D values by only taking 2 simulations from each of those setting pairs

selected_indices = {
    0, 1, 2,
    9, 10, 11,
    18, 19, 20,
    27, 28, 29,
    36, 37, 38,
    45, 46, 47,
    54, 55, 56, 57, 58,
    63, 64, 65, 66, 67, 68,
    72, 73, 74, 75, 76, 77, 78, 79}

filtered_data_list = []

for block_idx in range(81):
    start = block_idx * 12
    end = start + 12

    block = data_list_all[start:end]

    if block_idx in selected_indices:
        filtered_data_list.extend(copy.deepcopy(block[10:]))
    else:
        filtered_data_list.extend(copy.deepcopy(block))

print(f"Original length: {len(data_list_all)}")
print(f"Filtered length: {len(filtered_data_list)}")

Original length: 972
Filtered length: 602


In [7]:
# function for setting random seed number

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # GPU will be slower but deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False 
    print("Seeded everything: {}".format(seed))

In [8]:
# Get the statistics to normalize the input

def compute_stats(data_list):
    cell_area = []
    cell_perim = []
    cell_si = []
    nuc_area = []
    nuc_perim = []
    nuc_si = []
    diff = []
    edge_attr = []
    x_pos = []
    y_pos = []

    for data in data_list:
        cell_area.extend(data.x[:, 3])
        cell_perim.extend(data.x[:, 4])
        cell_si.extend(data.x[:, 5])
        nuc_area.extend(data.x[:, 0])
        nuc_perim.extend(data.x[:, 1])
        nuc_si.extend(data.x[:, 2])
        diff.extend([data.diffConst.item()])
        edge_attr.extend(data.edge_attr)
        x_pos.extend(data.cell_pos[:, 0])
        y_pos.extend(data.cell_pos[:, 1])

    stats = {
        "cell_area": (np.mean(cell_area), np.std(cell_area)),
        "cell_perim": (np.mean(cell_perim), np.std(cell_perim)),
        "cell_si": (np.mean(cell_si), np.std(cell_si)),
        "nuc_area": (np.mean(nuc_area), np.std(nuc_area)),
        "nuc_perim": (np.mean(nuc_perim), np.std(nuc_perim)),
        "nuc_si": (np.mean(nuc_si), np.std(nuc_si)),
        "diff": (np.mean(diff), np.std(diff)),
        "edge_attr": (np.mean(edge_attr), np.std(edge_attr)),
        "x_pos": (np.mean(x_pos), np.std(x_pos)),
        "y_pos": (np.mean(y_pos), np.std(y_pos)),
    }
    return stats

In [9]:
# function for picking diffusion constant for given snapshot as output

def setOutput(data_list):
    for data in data_list:
        data.y = data.diffConst
    return data_list

In [10]:
# function for choosing the node and edge information as the input

def setInput(data_list):
    for data in data_list:
        data.x = data.x
    return data_list

In [11]:
# Function to normalize all input data 

def normalize(data_list, stats):
    for data in data_list:
        
        # x features
        data.x[:, 3] = (data.x[:, 3] - stats["cell_area"][0]) / stats["cell_area"][1]
        data.x[:, 4] = (data.x[:, 4] - stats["cell_perim"][0]) / stats["cell_perim"][1]
        data.x[:, 5] = (data.x[:, 5] - stats["cell_si"][0]) / stats["cell_si"][1]

        data.x[:, 0] = (data.x[:, 0] - stats["nuc_area"][0]) / stats["nuc_area"][1]
        data.x[:, 1] = (data.x[:, 1] - stats["nuc_perim"][0]) / stats["nuc_perim"][1]
        data.x[:, 2] = (data.x[:, 2] - stats["nuc_si"][0]) / stats["nuc_si"][1]

        # other features
        data.diffConst = data.diffConst / stats["diff"][1]
        data.edge_attr = (data.edge_attr - stats["edge_attr"][0]) / stats["edge_attr"][1]
        data.cell_pos[:,0] = (data.cell_pos[:,0] - stats["x_pos"][0]) / stats["x_pos"][1]
        data.cell_pos[:,1] = (data.cell_pos[:,1] - stats["y_pos"][0]) / stats["y_pos"][1]

    return data_list

In [12]:
# Function to train the model and input area fraction as graph level

def train():
    model.train()
    for data in train_loader:
        optimizer.zero_grad()

        edge_attr = data.edge_attr.view(-1, 1)
        
        # calculate loss
        pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda(), data.area_frac.cuda())
        loss = criterion(pred.squeeze(), data.y.squeeze().cuda())

        # update
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    return pred.squeeze().detach().cpu(), data.y.squeeze()

In [13]:
@torch.no_grad()
def test(loader):
    model.eval()
    loss_list=[]
    pred_list=[]
    y_list=[]
    for data in loader:
        edge_attr = data.edge_attr.view(-1, 1)
        pred = model(data.x.cuda(), data.edge_index.cuda(), data.edge_attr.cuda(), data.batch.cuda(), data.area_frac.cuda())
        loss = criterion(pred.squeeze().detach().cpu(), data.y.squeeze())
        loss_list.append(loss.item())
        pred_list.extend(pred.squeeze().detach().cpu())
        y_list.extend(data.y.squeeze())
    return loss_list, pred_list, y_list

In [14]:
# Function to randomly select test and train indices from the 30 simulations out 300

def select_indices():

    # Run once when starting a whole new testing set 
    #selected_idx = np.random.choice(np.arange(len(filtered_data_list)), size=len(filtered_data_list), replace=False)
    #test_idx = np.random.choice(selected_idx, size=int(0.2*len(selected_idx)), replace=False)
    #np.save("TrainingResults_batch3_gridBig_[TrainStructure]/test_idx.npy", test_idx)

    test_idx = np.load("TrainingResults_batch3_gridBig_[TrainStructure]/test_idx.npy")

    # choose indices of preferred positions on grid out of original dataset (unfiltered)
    start_idx = [0, 24, 48, 72, 96, 
    	         216, 240, 264, 288, 312, 
                 432, 456, 480, 504, 528, 
                 648, 672, 696, 720, 744, 
                 864, 888, 912, 936, 960]
    train_idx = np.concatenate([np.arange(i, i + 12) for i in start_idx])
    

    val_idx = np.random.choice(np.arange(972), size=972, replace=False)

    test_idx_list = test_idx.tolist()
    train_idx_list = train_idx.tolist()
    val_idx_list = val_idx.tolist()

    return test_idx_list, train_idx_list, val_idx_list

In [15]:
# MAIN CODE WHERE TRAINING ACTUALLY HAPPENS

In [16]:
# create folders for saving results
os.makedirs('TrainingResults_batch3_gridBig_[TrainStructure]', exist_ok=True)
os.makedirs('TrainingWeights_batch3_gridBig_[TrainStructure]', exist_ok=True)

In [17]:
# rand seed initialization
set_seed(RandSeedNumber)

Seeded everything: 18264


In [18]:
# Select the indices of the simulations used for training and testing

test_idx, train_idx, val_idx = select_indices()

In [19]:
# Create the list with the actual data to train and test with

data_list_train = []
data_list_test = []
data_list_val = []

for i, item in enumerate(filtered_data_list):  # 602 folders
    if i in test_idx:
        data_list_test.extend(item)        

for i, item in enumerate(data_list_all):
    if i in train_idx:
        data_list_train.extend(item)

for i, item in enumerate(data_list_validation):  # 972 folders
    if i in val_idx:
        data_list_val.extend(item)   

print(f'Train size:{len(data_list_train)}, Test size:{len(data_list_test)}, Validation size:{len(data_list_val)}')

Train size:1500, Test size:600, Validation size:2916


In [20]:
# Compute mean and std for all features

stats = compute_stats(data_list_train)

In [21]:
# Obtain initial diffusion constant distribution of the test dataset

D_init = []
for data in data_list_test:
    D_init.append(data.diffConst)

plt.hist(D_init, bins=10, edgecolor='black')
plt.xlabel("D")
plt.ylabel("Counts")
plt.title("Initial D")

plt.savefig("TrainingResults_batch3_gridBig_[TrainStructure]/Hist_DinitTest_001.png", dpi=200, bbox_inches="tight")
plt.close()

In [22]:
# nondimesionalize lists

data_train_nondim = normalize(data_list_train, stats)
data_test_nondim = normalize(data_list_test, stats)
data_val_nondim = normalize(data_list_val, stats)

In [23]:
# Obtain diffusion constant distribution of the normalized train data

D_norm_train = []
for data in data_train_nondim:
    D_norm_train.append(data.diffConst)

plt.hist(D_norm_train, bins=10, edgecolor='black')
plt.xlabel("D")
plt.ylabel("Counts")
plt.title("Normalized D in train data")

plt.savefig(f"TrainingResults_batch3_gridBig_[TrainStructure]/Hist_DnormTrain_[{name}]_001.png", dpi=200, bbox_inches="tight")
plt.close()

In [24]:
# Shuffle the content of both lists since now all snapshots from same simulation are clustered together

random.shuffle(data_train_nondim)
random.shuffle(data_test_nondim)
random.shuffle(data_val_nondim)

In [25]:
# Select the diffusion constant as the output

data_train_nondim = setOutput(data_train_nondim)
data_test_nondim = setOutput(data_test_nondim)
data_val_nondim = setOutput(data_val_nondim)

In [26]:
# Select the node features and edges as the input 

data_train_nondim = setInput(data_train_nondim)
data_test_nondim = setInput(data_test_nondim)
data_val_nondim = setInput(data_val_nondim) 

In [27]:
# Visualize snapshots and graph representation of data

visualize_snapshot_cell(data_val_nondim, [50, 100])
visualize_snapshot_nucleus(data_val_nondim, [50, 100])
visualize_snapshot_graph(data_val_nondim, [50, 100])

In [28]:
# Settings for the model - MAIN VARIABLES

num_layers      = 5     
hidden_channels = 24     
batch_size      = 64   
learning_rate   = 1e-5   
weight_decay    = 1e-3   
EpochNum        = 201 

In [29]:
# Test what the capabilities of a linear regression model are

corr, MSE, p_value = lin_regression(data_list_val)
print(f'The correlation is: {corr} and the MSE is: {MSE}')

The correlation is: -0.4226669260066319 and the MSE is: 0.686991718458365


In [ ]:
g = torch.Generator()
g.manual_seed(RandSeedNumber)

In [30]:
# Use Dataloader to make data ready to train model with

train_loader = DataLoader(data_train_nondim, batch_size=batch_size, shuffle=True, generator=g, drop_last=True)
test_loader = DataLoader(data_test_nondim, batch_size=4096, shuffle=False, generator=g, drop_last=False) 

In [31]:
# initialize model

in_channels = data_list_train[0].x.shape[1] # Shape has (num_nodes, features) and you only want features
model = myPNA(data_list=data_train_nondim, in_channels=in_channels, hidden_channels=hidden_channels, num_layers=num_layers).cuda()
print(model)

tensor([    0,   290,  3782, 16046, 21593, 23452, 75579,  8983,   274,     1])
myPNA(
  (convs): ModuleList(
    (0): PNAConv(6, 24, towers=3, edge_dim=1)
    (1-4): 4 x PNAConv(24, 24, towers=3, edge_dim=1)
  )
  (batch_norms): ModuleList(
    (0-4): 5 x BatchNorm(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (lin): Linear(in_features=25, out_features=1, bias=True)
)


In [32]:
# initialize optimizer

criterion = torch.nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp) 

In [34]:
# train model
train_loss_lst=[]
test_loss_lst=[]
train_corr_lst=[]
test_corr_lst=[]
for epoch in range(1, EpochNum):
    pred, truth = train()

    with torch.no_grad():
        train_loss, train_pred, train_truth = test(train_loader)
        test_loss, test_pred, test_truth = test(test_loader)
        train_loss_lst.append(sum(train_loss)/len(train_loss))
        test_loss_lst.append(sum(test_loss)/len(test_loss))

        # calculate correlation
        train_corr, train_p_value = scipy.stats.pearsonr(np.array(train_pred),np.array(train_truth))
        test_corr, test_p_value = scipy.stats.pearsonr(np.array(test_pred),np.array(test_truth))

        train_corr_lst.append(train_corr)
        test_corr_lst.append(test_corr)

        print(f'Epoch: {epoch:03d}, Train loss: {sum(train_loss)/len(train_loss):.4f}, Test loss: {sum(test_loss)/len(test_loss):.4f},\
            Pearson correlation (train): {train_corr:.4f}, Pearson correlation (test): {test_corr:.4f}')

/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/home/20234017/.local/lib/python3.11/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch: 001, Train loss: 0.9730, Test loss: 1.0813,            Pearson correlation (train): 0.8168, Pearson correlation (test): 0.7603
Epoch: 002, Train loss: 0.8154, Test loss: 0.9127,            Pearson correlation (train): 0.8439, Pearson correlation (test): 0.8089
Epoch: 003, Train loss: 0.6701, Test loss: 0.7399,            Pearson correlation (train): 0.8690, Pearson correlation (test): 0.8244
Epoch: 004, Train loss: 0.5730, Test loss: 0.6234,            Pearson correlation (train): 0.8924, Pearson correlation (test): 0.8382
Epoch: 005, Train loss: 0.5032, Test loss: 0.5494,            Pearson correlation (train): 0.9038, Pearson correlation (test): 0.8454
Epoch: 006, Train loss: 0.4542, Test loss: 0.4854,            Pearson correlation (train): 0.9077, Pearson correlation (test): 0.8442
Epoch: 007, Train loss: 0.4233, Test loss: 0.4609,            Pearson correlation (train): 0.9106, Pearson correlation (test): 0.8474
Epoch: 008, Train loss: 0.3907, Test loss: 0.4264,            

In [35]:
# POST PROCESSING AND VISUALIZATION

In [36]:
# Plot the loss curves

def plot_losses(train, test):
    plt.figure()
    plt.semilogy(train, label="train", marker=".", lw=2)
    plt.semilogy(test, label="test", marker=".", lw=2)
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.savefig(f"TrainingResults_batch3_gridBig_[TrainStructure]/losses_[{name}]_{RandSeedNumber}_{nr}.png", dpi=200)
    plt.close()

plot_losses(train_loss_lst, test_loss_lst)

In [37]:
# Test the model on validation data (all the data)

val_loader = DataLoader(data_val_nondim, batch_size=2980, shuffle=False, generator=g, drop_last=False) 
val_loss, val_pred, val_truth = test(val_loader)

In [38]:
# Scale the model predictions back according to pre-processing

truth = np.array(val_truth) * stats["diff"][1] 
pred  = np.array(val_pred) * stats["diff"][1] 

In [39]:
print(pred.min())
print(truth.min())

0.37774703
0.011749101


In [40]:
# Obtain Pearson correlation of the validation dataset

val_corr, val_p_value = scipy.stats.pearsonr(np.array(pred),np.array(truth))
print(f'The Pearson correlation is: {val_corr}')
print(f'The p-value is: {val_p_value}')

The Pearson correlation is: 0.9348463259869441
The p-value is: 0.0


In [42]:
# obtain least-squares fit parameters

m, b = np.polyfit(truth, pred, 1)
print(f'The slope is: {m} and the intercept is: {b}')

The slope is: 0.7381285888687872 and the intercept is: 0.5673447458064772


In [43]:
# 2D probability density scatter plot of predictions

plt.figure(figsize=(4, 3))
ax = sns.kdeplot(x=np.array(truth), y=np.array(pred), cmap="rocket_r", fill=True, levels=10, cbar=True, cut=12)
plt.scatter(np.array(truth),np.array(pred), s=5, c='lightblue', alpha=0.6, label = 'Data')

# reference line
xref = np.linspace(0, 6.0, num=100)
plt.plot(xref,xref, c = 'k', label = 'Reference',linewidth = 1, linestyle = 'dashed')

# plot line with intercept
x = np.linspace(0, 6, 100)
y2 = m*x + b
plt.plot(x,y2, label = "Line", linewidth = 1)

plt.xlim([0,6.0])
plt.ylim([0,6.0])
plt.xlabel('$100D$',fontsize=12)
plt.ylabel('$100D_{NN}$',fontsize=12)
plt.tight_layout()
plt.savefig(f'TrainingResults_batch3_gridBig_[TrainStructure]/Fit_Line_noLog_[{name}]_{m:.3f}_{b:.3f}_{RandSeedNumber}_{nr}.png', dpi=300, bbox_inches='tight')
plt.close()

In [44]:
# save results

savePath = f'TrainingResults_batch3_gridBig_[TrainStructure]/GNNmodel_batch3_[{name}]_{nr}_layers{num_layers}_channels{hidden_channels}_batchSize{batch_size}_lr{learning_rate}_Epochs{EpochNum}_WeightDecay{weight_decay}_seed{RandSeedNumber}.mat'
io.savemat(savePath, {'train_loss': train_loss_lst,  'train_corr': train_corr_lst,
                      'test_loss' : test_loss_lst,   'test_corr': test_corr_lst})
print(savePath)

TrainingResults_batch3_gridBig_[TrainStructure]/GNNmodel_batch3_[25og]_005_layers5_channels24_batchSize64_lr1e-05_Epochs201_WeightDecay0.001_seed18264.mat


In [45]:
# save trained model weights

torch.save(model.state_dict(), f'TrainingWeights_batch3_gridBig_[TrainStructure]/GNNmodel_weights_[{name}]_{RandSeedNumber}_{nr}.pth')

In [46]:
# Plot average pearson correlations and their SEM

x_labels = ["4corner", "9sidesmiddle", "14highD", "13grid", "25og", "17cross"]
x = np.arange(len(x_labels))

values = np.array([0.7714, 0.9261, 0.9106, 0.9179, 0.9334, 0.6951])
std    = np.array([0.0068, 0.0011, 0.0085, 0.0068, 0.0016, 0.0065])

plt.figure(figsize=(5, 4))
ax = plt.gca()

plt.errorbar(x, values, yerr=std, fmt='o', capsize=4)

# ticks
plt.xticks(x, x_labels, rotation=30, ha='right')
ax.minorticks_on()
ax.tick_params(axis='x', which='minor', bottom=False)

plt.ylabel("Pearson correlation")

# grid (horizontal only)
ax.yaxis.grid(True, which='major', linewidth=0.5, alpha=0.3)
ax.yaxis.grid(True, which='minor', linewidth=0.3, alpha=0.15)
ax.xaxis.grid(False)

plt.tight_layout()

plt.savefig(
    "TrainingResults_batch3_gridBig_[TrainStructure]/TrainStructureResult_5seeds_001.png",
    dpi=300,
    bbox_inches="tight")

plt.close()

In [51]:
# plot the fits on Pearson colorscale

import matplotlib as mpl


slopes = np.array([
    [0.411784055, 0.494672083, 0.535543558, 0.529680358, 0.433676967],
    [0.763544378, 0.64681245, 0.715475996, 0.69587394, 0.725334156],
    [0.670594984, 0.477170061, 0.476086676, 0.703434587, 0.713030416],
    [0.647233976, 0.618726007, 0.451794969, 0.643616927, 0.722060622],
    [0.509314174, 0.65559459, 0.630356484, 0.670894375, 0.574456657],
    [0.695272899, 0.718858672, 0.761230811, 0.630315569, 0.738128589]
])

intercepts = np.array([
    [1.051969183, 1.070234887, 1.36482798, 1.190251578, 1.000781948],
    [0.677147424, 0.814374833, 0.638301805, 0.658378201, 0.732975535],
    [0.768714119, 0.522652338, 0.54239477, 0.603222235, 0.662018232],
    [0.537729334, 0.555610895, 0.483448756, 0.77040684, 0.489488961],
    [0.508710049, 0.65559459, 0.737200464, 0.671080735, 0.50771735],
    [0.455610975, 0.528187454, 0.369104977, 0.507773347, 0.567344746]
])

values = np.array([0.7714, 0.9261, 0.9106, 0.9179, 0.6951, 0.9334])

norm = mpl.colors.Normalize(vmin=values.min(), vmax=values.max())
cmap = plt.cm.viridis

slopes_g = slopes.reshape(6, 5)
intercepts_g = intercepts.reshape(6, 5)

mean_slopes = slopes_g.mean(axis=1)
mean_intercepts = intercepts_g.mean(axis=1)

labels = ["4corner", "9sidesmiddle", "14highD", "13grid", "17cross", "25og"]

x = np.linspace(0,6,100)

plt.figure(figsize=(5, 4))

fig, ax = plt.subplots(figsize=(5, 4))

for i, (m, b, r) in enumerate(zip(
        mean_slopes, mean_intercepts,
        values)):

    color = cmap(norm(r))

    y_mean = m * x + b

    ax.plot(x, y_mean, color=color, linewidth=0.8, label=labels[i])

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])  

cbar = fig.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Pearson correlation")

# reference line
xref = np.linspace(0, 6.0, 100)
plt.plot(xref, xref, c='k', linestyle='dashed', linewidth=1, label='Reference')

plt.grid(True, which='major', linestyle='-', linewidth=1.0, alpha=0.3)
plt.minorticks_on()
plt.grid(True, which='minor', linestyle=':', linewidth=0.7, alpha=0.15)

plt.xlim([0, 6.0])
plt.ylim([0, 6.0])

plt.xlabel(r'$100D$', fontsize=12)
plt.ylabel(r'$100D_{NN}$', fontsize=12)

plt.legend(
    fontsize=8,
    labelspacing=0.2,
    handlelength=1.2,
    handletextpad=0.4,
    ncol=2)

plt.tight_layout()
plt.savefig(
    'TrainingResults_batch3_gridBig_[TrainStructure]/AllFits_TStruct_[correctAx5seeds]_003.png',
    dpi=300,
    bbox_inches='tight')

plt.close()

In [50]:
# statistical ANOVA tests to see significance of results 

from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

g1 = [0.789429203, 0.77010254, 0.751647962, 0.76277676, 0.783088349]
g2 = [0.929538223, 0.923460814, 0.923705315, 0.926776848, 0.926989784]
g3 = [0.923573591, 0.894981976, 0.885675581, 0.927571342, 0.921422679]
g4 = [0.925096252, 0.921853323, 0.890913235, 0.925569492, 0.926024064]
g5 = [0.690961336, 0.68783751, 0.720879957, 0.688724712, 0.687085679]
g6 = [0.936403352, 0.928002747, 0.936397605, 0.93159558, 0.934846326]

# ANOVA
F, p = f_oneway(g1, g2, g3, g4, g5, g6)

print("ANOVA:")
print("F =", F)
print("p =", p)

# TUKEY HSD
data = g1 + g2 + g3 + g4 + g5 + g6
labels = (
    ["4corner"]*5 +
    ["9sidesmiddle"]*5 +
    ["14highD"]*5 +
    ["13grid"]*5 +
    ["17cross"]*5 +
    ["25og"]*5
)

tukey = pairwise_tukeyhsd(endog=data, groups=labels, alpha=0.05)
print(tukey)

ANOVA:
F = 289.4884247808735
p = 1.2566914689872799e-20
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1    group2    meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
 13grid      14highD  -0.0072 0.9511 -0.0331  0.0186  False
 13grid      17cross  -0.2228    0.0 -0.2487 -0.1969   True
 13grid         25og   0.0156 0.4499 -0.0103  0.0414  False
 13grid      4corner  -0.1465    0.0 -0.1724 -0.1206   True
 13grid 9sidesmiddle   0.0082 0.9199 -0.0177  0.0341  False
14highD      17cross  -0.2155    0.0 -0.2414 -0.1897   True
14highD         25og   0.0228 0.1069 -0.0031  0.0487  False
14highD      4corner  -0.1392    0.0 -0.1651 -0.1134   True
14highD 9sidesmiddle   0.0154 0.4575 -0.0104  0.0413  False
17cross         25og   0.2384    0.0  0.2125  0.2642   True
17cross      4corner   0.0763    0.0  0.0504  0.1022   True
17cross 9sidesmiddle    0.231    0.0  0.2051  0.2569   True
   25og      4corner   -0.162    0.0 -0.1879